In [0]:
{
  "datasets": [
    {
      "name": "61afe924",
      "displayName": "Top 10 Products by Revenue",
      "queryLines": [
        "\n",
        "SELECT \n",
        "  p.`product_name`,\n",
        "  p.`category`,\n",
        "  p.`subcategory`,\n",
        "  SUM(f.`sales_amount`) AS total_revenue,\n",
        "  SUM(f.`quantity`) AS total_quantity_sold,\n",
        "  ROUND(SUM(f.`sales_amount`) / SUM(f.`quantity`), 2) AS avg_price_per_unit,\n",
        "  COUNT(DISTINCT f.`order_number`) AS order_count\n",
        "FROM `workspace`.`gold`.`fact_sales` f\n",
        "JOIN `workspace`.`gold`.`dim_products` p ON f.`product_key` = p.`product_key`\n",
        "WHERE f.`order_date` IS NOT NULL\n",
        "  AND p.`product_name` IS NOT NULL\n",
        "GROUP BY p.`product_name`, p.`category`, p.`subcategory`\n",
        "ORDER BY total_revenue DESC\n",
        "LIMIT 10\n"
      ]
    },
    {
      "name": "51a7514e",
      "displayName": "Monthly Revenue and Profit Trend",
      "queryLines": [
        "\n",
        "SELECT \n",
        "  DATE_TRUNC('month', `order_date`) AS order_month,\n",
        "  SUM(`sales_amount`) AS total_revenue,\n",
        "  SUM(`sales_amount`) AS total_profit\n",
        "FROM `workspace`.`gold`.`fact_sales`\n",
        "WHERE `order_date` IS NOT NULL\n",
        "GROUP BY DATE_TRUNC('month', `order_date`)\n",
        "ORDER BY order_month\n"
      ]
    },
    {
      "name": "11a3f2a6",
      "displayName": "Sales Distribution by Customer Segment (Geographic)",
      "queryLines": [
        "\n",
        "-- Create customer segments based on country (as a proxy for Enterprise/SMB/Consumer)\n",
        "-- Using country as the primary segmentation dimension\n",
        "SELECT \n",
        "  CASE \n",
        "    WHEN c.`country` = 'United States' THEN 'North America'\n",
        "    WHEN c.`country` IN ('United Kingdom', 'France', 'Germany') THEN 'Europe'\n",
        "    WHEN c.`country` IN ('Australia', 'Canada') THEN 'Pacific & Canada'\n",
        "    WHEN c.`country` = 'n/a' THEN 'Unknown/Online'\n",
        "    ELSE 'Other Markets'\n",
        "  END AS customer_segment,\n",
        "  SUM(f.`sales_amount`) AS total_sales_amount,\n",
        "  COUNT(DISTINCT f.`order_number`) AS order_count,\n",
        "  COUNT(DISTINCT f.`customer_key`) AS customer_count,\n",
        "  ROUND(100.0 * SUM(f.`sales_amount`) / SUM(SUM(f.`sales_amount`)) OVER (), 2) AS sales_percentage\n",
        "FROM `workspace`.`gold`.`fact_sales` f\n",
        "JOIN `workspace`.`gold`.`dim_customers` c ON f.`customer_key` = c.`customer_key`\n",
        "WHERE f.`order_date` IS NOT NULL\n",
        "  AND c.`country` IS NOT NULL\n",
        "GROUP BY \n",
        "  CASE \n",
        "    WHEN c.`country` = 'United States' THEN 'North America'\n",
        "    WHEN c.`country` IN ('United Kingdom', 'France', 'Germany') THEN 'Europe'\n",
        "    WHEN c.`country` IN ('Australia', 'Canada') THEN 'Pacific & Canada'\n",
        "    WHEN c.`country` = 'n/a' THEN 'Unknown/Online'\n",
        "    ELSE 'Other Markets'\n",
        "  END\n",
        "ORDER BY total_sales_amount DESC\n"
      ]
    },
    {
      "name": "5bbadc36",
      "displayName": "Sales Distribution by Customer Demographics",
      "queryLines": [
        "\n",
        "-- Alternative segmentation: By customer demographics (marital status + gender)\n",
        "SELECT \n",
        "  CASE \n",
        "    WHEN c.`marital_status` = 'Married' AND c.`gender` = 'Male' THEN 'Married Male'\n",
        "    WHEN c.`marital_status` = 'Married' AND c.`gender` = 'Female' THEN 'Married Female'\n",
        "    WHEN c.`marital_status` = 'Single' AND c.`gender` = 'Male' THEN 'Single Male'\n",
        "    WHEN c.`marital_status` = 'Single' AND c.`gender` = 'Female' THEN 'Single Female'\n",
        "    ELSE 'Unknown'\n",
        "  END AS customer_segment,\n",
        "  SUM(f.`sales_amount`) AS total_sales_amount,\n",
        "  COUNT(DISTINCT f.`order_number`) AS order_count,\n",
        "  COUNT(DISTINCT f.`customer_key`) AS customer_count,\n",
        "  ROUND(100.0 * SUM(f.`sales_amount`) / SUM(SUM(f.`sales_amount`)) OVER (), 2) AS sales_percentage\n",
        "FROM `workspace`.`gold`.`fact_sales` f\n",
        "JOIN `workspace`.`gold`.`dim_customers` c ON f.`customer_key` = c.`customer_key`\n",
        "WHERE f.`order_date` IS NOT NULL\n",
        "GROUP BY \n",
        "  CASE \n",
        "    WHEN c.`marital_status` = 'Married' AND c.`gender` = 'Male' THEN 'Married Male'\n",
        "    WHEN c.`marital_status` = 'Married' AND c.`gender` = 'Female' THEN 'Married Female'\n",
        "    WHEN c.`marital_status` = 'Single' AND c.`gender` = 'Male' THEN 'Single Male'\n",
        "    WHEN c.`marital_status` = 'Single' AND c.`gender` = 'Female' THEN 'Single Female'\n",
        "    ELSE 'Unknown'\n",
        "  END\n",
        "ORDER BY total_sales_amount DESC\n"
      ]
    },
    {
      "name": "d7848ec8",
      "displayName": "Sales Performance by Region and Product Category",
      "queryLines": [
        "\n",
        "-- Sales by region and category, pre-sorted by total region revenue (highest to lowest)\n",
        "WITH region_totals AS (\n",
        "  SELECT \n",
        "    c.`country` AS region,\n",
        "    SUM(f.`sales_amount`) AS region_total_revenue\n",
        "  FROM `workspace`.`gold`.`fact_sales` f\n",
        "  JOIN `workspace`.`gold`.`dim_customers` c ON f.`customer_key` = c.`customer_key`\n",
        "  WHERE f.`order_date` IS NOT NULL\n",
        "    AND c.`country` IS NOT NULL\n",
        "    AND c.`country` != 'n/a'\n",
        "  GROUP BY c.`country`\n",
        "),\n",
        "region_category_sales AS (\n",
        "  SELECT \n",
        "    c.`country` AS region,\n",
        "    p.`category`,\n",
        "    SUM(f.`sales_amount`) AS total_revenue,\n",
        "    rt.region_total_revenue\n",
        "  FROM `workspace`.`gold`.`fact_sales` f\n",
        "  JOIN `workspace`.`gold`.`dim_customers` c ON f.`customer_key` = c.`customer_key`\n",
        "  JOIN `workspace`.`gold`.`dim_products` p ON f.`product_key` = p.`product_key`\n",
        "  JOIN region_totals rt ON c.`country` = rt.region\n",
        "  WHERE f.`order_date` IS NOT NULL\n",
        "    AND c.`country` IS NOT NULL\n",
        "    AND c.`country` != 'n/a'\n",
        "    AND p.`category` IS NOT NULL\n",
        "  GROUP BY c.`country`, p.`category`, rt.region_total_revenue\n",
        ")\n",
        "SELECT \n",
        "  region,\n",
        "  category,\n",
        "  total_revenue\n",
        "FROM region_category_sales\n",
        "ORDER BY region_total_revenue DESC, region, category\n"
      ]
    },
    {
      "name": "4becc378",
      "displayName": "2013 Business Performance - KPI Summary Cards",
      "queryLines": [
        "\n",
        "-- Format KPI metrics as individual cards with YoY comparison\n",
        "WITH kpi_data AS (\n",
        "  SELECT \n",
        "    cy.total_revenue AS current_revenue,\n",
        "    cy.total_profit AS current_profit,\n",
        "    cy.total_orders AS current_orders,\n",
        "    cy.avg_order_value AS current_aov,\n",
        "    py.total_revenue AS prior_revenue,\n",
        "    py.total_profit AS prior_profit,\n",
        "    py.total_orders AS prior_orders,\n",
        "    py.avg_order_value AS prior_aov,\n",
        "    ROUND(((cy.total_revenue - py.total_revenue) / py.total_revenue) * 100, 2) AS revenue_yoy_pct,\n",
        "    ROUND(((cy.total_profit - py.total_profit) / py.total_profit) * 100, 2) AS profit_yoy_pct,\n",
        "    ROUND(((cy.total_orders - py.total_orders) / py.total_orders) * 100, 2) AS orders_yoy_pct,\n",
        "    ROUND(((cy.avg_order_value - py.avg_order_value) / py.avg_order_value) * 100, 2) AS aov_yoy_pct\n",
        "  FROM (\n",
        "    SELECT \n",
        "      SUM(`sales_amount`) AS total_revenue,\n",
        "      SUM(`sales_amount`) AS total_profit,\n",
        "      COUNT(DISTINCT `order_number`) AS total_orders,\n",
        "      ROUND(SUM(`sales_amount`) / COUNT(DISTINCT `order_number`), 2) AS avg_order_value\n",
        "    FROM `workspace`.`gold`.`fact_sales`\n",
        "    WHERE YEAR(`order_date`) = 2013\n",
        "  ) cy\n",
        "  CROSS JOIN (\n",
        "    SELECT \n",
        "      SUM(`sales_amount`) AS total_revenue,\n",
        "      SUM(`sales_amount`) AS total_profit,\n",
        "      COUNT(DISTINCT `order_number`) AS total_orders,\n",
        "      ROUND(SUM(`sales_amount`) / COUNT(DISTINCT `order_number`), 2) AS avg_order_value\n",
        "    FROM `workspace`.`gold`.`fact_sales`\n",
        "    WHERE YEAR(`order_date`) = 2012\n",
        "  ) py\n",
        ")\n",
        "SELECT \n",
        "  'Total Revenue' AS kpi_name,\n",
        "  CONCAT('$', FORMAT_NUMBER(current_revenue, '#,##0')) AS current_value,\n",
        "  CONCAT(\n",
        "    CASE WHEN revenue_yoy_pct > 0 THEN '↑ +' ELSE '↓ ' END,\n",
        "    FORMAT_NUMBER(revenue_yoy_pct, '#,##0.0'), '%'\n",
        "  ) AS yoy_change,\n",
        "  'vs 2012' AS comparison_period\n",
        "FROM kpi_data\n",
        "\n",
        "UNION ALL\n",
        "\n",
        "SELECT \n",
        "  'Total Profit' AS kpi_name,\n",
        "  CONCAT('$', FORMAT_NUMBER(current_profit, '#,##0')) AS current_value,\n",
        "  CONCAT(\n",
        "    CASE WHEN profit_yoy_pct > 0 THEN '↑ +' ELSE '↓ ' END,\n",
        "    FORMAT_NUMBER(profit_yoy_pct, '#,##0.0'), '%'\n",
        "  ) AS yoy_change,\n",
        "  'vs 2012' AS comparison_period\n",
        "FROM kpi_data\n",
        "\n",
        "UNION ALL\n",
        "\n",
        "SELECT \n",
        "  'Total Orders' AS kpi_name,\n",
        "  FORMAT_NUMBER(current_orders, '#,##0') AS current_value,\n",
        "  CONCAT(\n",
        "    CASE WHEN orders_yoy_pct > 0 THEN '↑ +' ELSE '↓ ' END,\n",
        "    FORMAT_NUMBER(orders_yoy_pct, '#,##0.0'), '%'\n",
        "  ) AS yoy_change,\n",
        "  'vs 2012' AS comparison_period\n",
        "FROM kpi_data\n",
        "\n",
        "UNION ALL\n",
        "\n",
        "SELECT \n",
        "  'Average Order Value' AS kpi_name,\n",
        "  CONCAT('$', FORMAT_NUMBER(current_aov, '#,##0.00')) AS current_value,\n",
        "  CONCAT(\n",
        "    CASE WHEN aov_yoy_pct > 0 THEN '↑ +' ELSE '↓ ' END,\n",
        "    FORMAT_NUMBER(ABS(aov_yoy_pct), '#,##0.0'), '%'\n",
        "  ) AS yoy_change,\n",
        "  'vs 2012' AS comparison_period\n",
        "FROM kpi_data\n"
      ]
    },
    {
      "name": "22f9dace",
      "displayName": "Total Revenue",
      "queryLines": [
        "\n",
        "-- Card 1: Total Revenue\n",
        "WITH kpi_data AS (\n",
        "  SELECT \n",
        "    cy.total_revenue AS current_value,\n",
        "    py.total_revenue AS prior_value,\n",
        "    ROUND(((cy.total_revenue - py.total_revenue) / py.total_revenue) * 100, 2) AS yoy_pct\n",
        "  FROM (\n",
        "    SELECT SUM(`sales_amount`) AS total_revenue\n",
        "    FROM `workspace`.`gold`.`fact_sales`\n",
        "    WHERE YEAR(`order_date`) = 2013\n",
        "  ) cy\n",
        "  CROSS JOIN (\n",
        "    SELECT SUM(`sales_amount`) AS total_revenue\n",
        "    FROM `workspace`.`gold`.`fact_sales`\n",
        "    WHERE YEAR(`order_date`) = 2012\n",
        "  ) py\n",
        ")\n",
        "SELECT \n",
        "  'Total Revenue' AS metric_name,\n",
        "  current_value,\n",
        "  yoy_pct,\n",
        "  CONCAT(CASE WHEN yoy_pct > 0 THEN '↑ +' ELSE '↓ ' END, FORMAT_NUMBER(yoy_pct, '#,##0.0'), '% vs 2012') AS yoy_label\n",
        "FROM kpi_data\n"
      ]
    },
    {
      "name": "7a412b8c",
      "displayName": "Total Profit",
      "queryLines": [
        "\n",
        "-- Card 2: Total Profit\n",
        "WITH kpi_data AS (\n",
        "  SELECT \n",
        "    cy.total_profit AS current_value,\n",
        "    py.total_profit AS prior_value,\n",
        "    ROUND(((cy.total_profit - py.total_profit) / py.total_profit) * 100, 2) AS yoy_pct\n",
        "  FROM (\n",
        "    SELECT SUM(`sales_amount`) AS total_profit\n",
        "    FROM `workspace`.`gold`.`fact_sales`\n",
        "    WHERE YEAR(`order_date`) = 2013\n",
        "  ) cy\n",
        "  CROSS JOIN (\n",
        "    SELECT SUM(`sales_amount`) AS total_profit\n",
        "    FROM `workspace`.`gold`.`fact_sales`\n",
        "    WHERE YEAR(`order_date`) = 2012\n",
        "  ) py\n",
        ")\n",
        "SELECT \n",
        "  'Total Profit' AS metric_name,\n",
        "  current_value,\n",
        "  yoy_pct,\n",
        "  CONCAT(CASE WHEN yoy_pct > 0 THEN '↑ +' ELSE '↓ ' END, FORMAT_NUMBER(yoy_pct, '#,##0.0'), '% vs 2012') AS yoy_label\n",
        "FROM kpi_data\n"
      ]
    },
    {
      "name": "e059b066",
      "displayName": "Total Orders",
      "queryLines": [
        "\n",
        "-- Card 3: Total Orders\n",
        "WITH kpi_data AS (\n",
        "  SELECT \n",
        "    cy.total_orders AS current_value,\n",
        "    py.total_orders AS prior_value,\n",
        "    ROUND(((cy.total_orders - py.total_orders) / CAST(py.total_orders AS DOUBLE)) * 100, 2) AS yoy_pct\n",
        "  FROM (\n",
        "    SELECT COUNT(DISTINCT `order_number`) AS total_orders\n",
        "    FROM `workspace`.`gold`.`fact_sales`\n",
        "    WHERE YEAR(`order_date`) = 2013\n",
        "  ) cy\n",
        "  CROSS JOIN (\n",
        "    SELECT COUNT(DISTINCT `order_number`) AS total_orders\n",
        "    FROM `workspace`.`gold`.`fact_sales`\n",
        "    WHERE YEAR(`order_date`) = 2012\n",
        "  ) py\n",
        ")\n",
        "SELECT \n",
        "  'Total Orders' AS metric_name,\n",
        "  current_value,\n",
        "  yoy_pct,\n",
        "  CONCAT(CASE WHEN yoy_pct > 0 THEN '↑ +' ELSE '↓ ' END, FORMAT_NUMBER(yoy_pct, '#,##0.0'), '% vs 2012') AS yoy_label\n",
        "FROM kpi_data\n"
      ]
    },
    {
      "name": "051c5d4a",
      "displayName": "Average Order Value (AOV)",
      "queryLines": [
        "\n",
        "-- Card 4: Average Order Value\n",
        "WITH kpi_data AS (\n",
        "  SELECT \n",
        "    cy.avg_order_value AS current_value,\n",
        "    py.avg_order_value AS prior_value,\n",
        "    ROUND(((cy.avg_order_value - py.avg_order_value) / py.avg_order_value) * 100, 2) AS yoy_pct\n",
        "  FROM (\n",
        "    SELECT ROUND(SUM(`sales_amount`) / COUNT(DISTINCT `order_number`), 2) AS avg_order_value\n",
        "    FROM `workspace`.`gold`.`fact_sales`\n",
        "    WHERE YEAR(`order_date`) = 2013\n",
        "  ) cy\n",
        "  CROSS JOIN (\n",
        "    SELECT ROUND(SUM(`sales_amount`) / COUNT(DISTINCT `order_number`), 2) AS avg_order_value\n",
        "    FROM `workspace`.`gold`.`fact_sales`\n",
        "    WHERE YEAR(`order_date`) = 2012\n",
        "  ) py\n",
        ")\n",
        "SELECT \n",
        "  'Average Order Value' AS metric_name,\n",
        "  current_value,\n",
        "  yoy_pct,\n",
        "  CONCAT(CASE WHEN yoy_pct > 0 THEN '↑ +' ELSE '↓ ' END, FORMAT_NUMBER(ABS(yoy_pct), '#,##0.0'), '% vs 2012') AS yoy_label\n",
        "FROM kpi_data\n"
      ]
    }
  ],
  "pages": [
    {
      "name": "e29e8e52",
      "displayName": "executive_sales_dashboard",
      "layout": [
        {
          "widget": {
            "name": "b992e9db",
            "queries": [
              {
                "name": "main_query",
                "query": {
                  "datasetName": "61afe924",
                  "fields": [
                    {
                      "name": "total_revenue",
                      "expression": "`total_revenue`"
                    },
                    {
                      "name": "product_name",
                      "expression": "`product_name`"
                    }
                  ],
                  "disaggregated": true
                }
              }
            ],
            "spec": {
              "frame": {
                "showTitle": true,
                "title": {
                  "value": "<span style=\"color: #07079D;\">Top 10 Products by Revenue</span>",
                  "fields": []
                }
              },
              "version": 3,
              "mark": {
                "colors": [
                  {
                    "themeColorType": "visualizationColors",
                    "position": 5
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 2
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 3
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 4
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 5
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 6
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 7
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 8
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 9
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 10
                  }
                ]
              },
              "widgetType": "bar",
              "encodings": {
                "x": {
                  "fieldName": "total_revenue",
                  "format": {
                    "type": "number-currency",
                    "currencyCode": "USD",
                    "abbreviation": "compact",
                    "decimalPlaces": {
                      "type": "exact",
                      "places": 1
                    }
                  },
                  "displayName": "Total Revenue",
                  "scale": {
                    "type": "quantitative"
                  }
                },
                "y": {
                  "fieldName": "product_name",
                  "displayName": "Product Name",
                  "scale": {
                    "type": "categorical",
                    "sort": {
                      "by": "custom-order",
                      "orderedValues": [
                        "Mountain-200 Black- 46",
                        "Mountain-200 Black- 42",
                        "Mountain-200 Silver- 38",
                        "Mountain-200 Silver- 46",
                        "Mountain-200 Black- 38",
                        "Mountain-200 Silver- 42",
                        "Road-250 Black- 52",
                        "Road-250 Red- 58",
                        "Road-250 Black- 48",
                        "Road-250 Black- 44"
                      ]
                    }
                  }
                },
                "label": {
                  "show": true
                }
              },
              "data": {
                "queryName": "main_query"
              }
            }
          },
          "position": {
            "x": 0,
            "y": 8,
            "width": 7,
            "height": 6
          }
        },
        {
          "widget": {
            "name": "b5622895",
            "queries": [
              {
                "name": "main_query",
                "query": {
                  "datasetName": "11a3f2a6",
                  "fields": [
                    {
                      "name": "total_sales_amount",
                      "expression": "`total_sales_amount`"
                    },
                    {
                      "name": "customer_segment",
                      "expression": "`customer_segment`"
                    }
                  ],
                  "disaggregated": true
                }
              }
            ],
            "spec": {
              "frame": {
                "showTitle": true,
                "title": {
                  "value": "<span style=\"color: #07079D;\">Sales Distribution by Customer Segment (Geographic)</span>",
                  "fields": []
                }
              },
              "mark": {
                "layout": "stack"
              },
              "version": 3,
              "widgetType": "pie",
              "encodings": {
                "angle": {
                  "fieldName": "total_sales_amount",
                  "scale": {
                    "type": "quantitative"
                  },
                  "format": {
                    "type": "number-currency",
                    "currencyCode": "USD",
                    "abbreviation": "compact",
                    "decimalPlaces": {
                      "type": "exact",
                      "places": 1
                    }
                  },
                  "displayName": "Total Sales"
                },
                "color": {
                  "fieldName": "customer_segment",
                  "legend": {
                    "position": "right"
                  },
                  "displayName": "Customer Segment",
                  "scale": {
                    "type": "categorical",
                    "sort": {
                      "by": "custom-order",
                      "orderedValues": [
                        "Pacific & Canada",
                        "Europe",
                        "North America",
                        "Unknown/Online"
                      ]
                    },
                    "mappings": [
                      {
                        "value": "Pacific & Canada",
                        "color": "#1E3A8A"
                      },
                      {
                        "value": "Europe",
                        "color": "#25D1EB"
                      },
                      {
                        "value": "North America",
                        "color": "#0EA5E9"
                      },
                      {
                        "value": "Unknown/Online",
                        "color": "#EC4899"
                      }
                    ]
                  }
                },
                "label": {
                  "show": true
                }
              },
              "data": {
                "queryName": "main_query"
              }
            }
          },
          "position": {
            "x": 8,
            "y": 2,
            "width": 4,
            "height": 6
          }
        },
        {
          "widget": {
            "name": "c02af86f",
            "queries": [
              {
                "name": "main_query",
                "query": {
                  "datasetName": "51a7514e",
                  "fields": [
                    {
                      "name": "order_month",
                      "expression": "`order_month`"
                    },
                    {
                      "name": "total_revenue",
                      "expression": "`total_revenue`"
                    },
                    {
                      "name": "total_profit",
                      "expression": "`total_profit`"
                    }
                  ],
                  "disaggregated": true
                }
              }
            ],
            "spec": {
              "frame": {
                "showTitle": true,
                "showDescription": false,
                "title": {
                  "value": "<span style=\"color: #07079D;\">Monthly Revenue and Profit Trend</span>",
                  "fields": []
                }
              },
              "version": 3,
              "mark": {
                "marker": {
                  "shape": "circle"
                },
                "colors": [
                  "#2563EB",
                  "#EC4899",
                  {
                    "themeColorType": "visualizationColors",
                    "position": 3
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 4
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 5
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 6
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 7
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 8
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 9
                  },
                  {
                    "themeColorType": "visualizationColors",
                    "position": 10
                  }
                ],
                "lineShape": "linear",
                "linePatterns": [
                  "solid"
                ]
              },
              "widgetType": "line",
              "encodings": {
                "x": {
                  "fieldName": "order_month",
                  "displayName": "Order Date",
                  "scale": {
                    "type": "temporal"
                  }
                },
                "y": {
                  "primary": {
                    "fields": [
                      {
                        "fieldName": "total_revenue",
                        "displayName": "Total Revenue",
                        "error": {
                          "type": "absolute"
                        }
                      }
                    ],
                    "scale": {
                      "type": "quantitative"
                    },
                    "format": {
                      "type": "number-currency",
                      "currencyCode": "USD",
                      "abbreviation": "compact",
                      "decimalPlaces": {
                        "type": "exact",
                        "places": 1
                      }
                    },
                    "axis": {
                      "title": "Total Revenue (USD)"
                    }
                  },
                  "secondary": {
                    "fields": [
                      {
                        "fieldName": "total_profit",
                        "displayName": "Total Profit"
                      }
                    ],
                    "scale": {
                      "type": "quantitative"
                    },
                    "format": {
                      "type": "number-currency",
                      "currencyCode": "USD",
                      "abbreviation": "compact",
                      "decimalPlaces": {
                        "type": "exact",
                        "places": 1
                      }
                    },
                    "axis": {
                      "title": "Total Profit (USD)"
                    }
                  }
                },
                "label": {
                  "show": false
                }
              },
              "data": {
                "queryName": "main_query"
              }
            },
            "specExtensions": {
              "widgetBorderColor": {
                "light": "#F1F5F9"
              }
            }
          },
          "position": {
            "x": 0,
            "y": 2,
            "width": 8,
            "height": 6
          }
        },
        {
          "widget": {
            "name": "64daabc4",
            "queries": [
              {
                "name": "main_query",
                "query": {
                  "datasetName": "d7848ec8",
                  "fields": [
                    {
                      "name": "category",
                      "expression": "`category`"
                    },
                    {
                      "name": "region",
                      "expression": "`region`"
                    },
                    {
                      "name": "total_revenue",
                      "expression": "`total_revenue`"
                    }
                  ],
                  "disaggregated": true
                }
              }
            ],
            "spec": {
              "frame": {
                "showTitle": true,
                "title": {
                  "value": "<span style=\"color: #07079D;\">Sales Performance by Region and Product Category</span>",
                  "fields": []
                },
                "description": {
                  "value": "Regions sorted from highest to lowest total revenue",
                  "fields": []
                }
              },
              "version": 3,
              "mark": {
                "layout": "stack"
              },
              "widgetType": "bar",
              "encodings": {
                "x": {
                  "fieldName": "region",
                  "displayName": "Customer Region",
                  "scale": {
                    "type": "categorical",
                    "sort": {
                      "by": "custom-order",
                      "orderedValues": [
                        "Australia",
                        "United States",
                        "United Kingdom",
                        "Germany",
                        "France",
                        "Canada"
                      ]
                    }
                  }
                },
                "y": {
                  "fieldName": "total_revenue",
                  "format": {
                    "type": "number-currency",
                    "currencyCode": "USD",
                    "abbreviation": "compact",
                    "decimalPlaces": {
                      "type": "exact",
                      "places": 1
                    }
                  },
                  "axis": {
                    "title": "Total Revenue (USD)"
                  },
                  "displayName": "Total Revenue",
                  "scale": {
                    "type": "quantitative"
                  }
                },
                "color": {
                  "fieldName": "category",
                  "legend": {
                    "position": "right"
                  },
                  "displayName": "Product Category",
                  "scale": {
                    "type": "categorical",
                    "sort": {
                      "by": "custom-order",
                      "orderedValues": [
                        "Accessories",
                        "Bikes",
                        "Clothing"
                      ]
                    },
                    "mappings": [
                      {
                        "value": "Bikes",
                        "color": "#D89FBE"
                      },
                      {
                        "value": "Accessories",
                        "color": {
                          "themeColorType": "selectionColor"
                        }
                      },
                      {
                        "value": "Clothing",
                        "color": "#323948"
                      }
                    ]
                  }
                },
                "label": {
                  "show": true
                }
              },
              "data": {
                "queryName": "main_query"
              }
            }
          },
          "position": {
            "x": 7,
            "y": 8,
            "width": 5,
            "height": 6
          }
        },
        {
          "widget": {
            "name": "3ac26bad",
            "queries": [
              {
                "name": "main_query",
                "query": {
                  "datasetName": "22f9dace",
                  "fields": [
                    {
                      "name": "yoy_pct",
                      "expression": "`yoy_pct`"
                    },
                    {
                      "name": "current_value",
                      "expression": "`current_value`"
                    }
                  ],
                  "disaggregated": true,
                  "orders": [
                    {
                      "direction": "DESC",
                      "expression": "`yoy_pct`"
                    }
                  ]
                }
              }
            ],
            "spec": {
              "frame": {
                "showTitle": true,
                "showDescription": true,
                "title": {
                  "value": "<span style=\"color: #07079D;\">Total Revenue</span>",
                  "fields": []
                },
                "description": {
                  "value": "2013 | YoY vs 2012",
                  "fields": []
                }
              },
              "version": 2,
              "widgetType": "counter",
              "encodings": {
                "value": {
                  "fieldName": "current_value"
                },
                "period": {
                  "fieldName": "yoy_pct",
                  "format": {
                    "type": "number-percent",
                    "decimalPlaces": {
                      "type": "max",
                      "places": 2
                    },
                    "negativeSign": "standard"
                  },
                  "style": {
                    "fontWeight": 400,
                    "fontStyle": "normal",
                    "fontColor": {
                      "themeColorType": "visualizationColors",
                      "position": 3
                    }
                  }
                }
              },
              "data": {
                "queryName": "main_query"
              }
            },
            "specExtensions": {
              "widgetBackgroundColor": {
                "light": "#FCE7F3"
              },
              "widgetBorderColor": {
                "light": "#D843A7"
              }
            }
          },
          "position": {
            "x": 0,
            "y": 0,
            "width": 3,
            "height": 2
          }
        },
        {
          "widget": {
            "name": "b051cf6d",
            "queries": [
              {
                "name": "main_query",
                "query": {
                  "datasetName": "7a412b8c",
                  "fields": [
                    {
                      "name": "yoy_pct",
                      "expression": "`yoy_pct`"
                    },
                    {
                      "name": "current_value",
                      "expression": "`current_value`"
                    }
                  ],
                  "disaggregated": true,
                  "orders": [
                    {
                      "direction": "DESC",
                      "expression": "`yoy_pct`"
                    }
                  ]
                }
              }
            ],
            "spec": {
              "frame": {
                "showTitle": true,
                "showDescription": true,
                "title": {
                  "value": "<span style=\"color: #07079D;\">Total Profit</span>",
                  "fields": []
                },
                "description": {
                  "value": "2013 | YoY vs 2012",
                  "fields": []
                }
              },
              "version": 2,
              "widgetType": "counter",
              "encodings": {
                "value": {
                  "fieldName": "current_value"
                },
                "period": {
                  "fieldName": "yoy_pct",
                  "style": {
                    "fontColor": {
                      "themeColorType": "visualizationColors",
                      "position": 3
                    }
                  },
                  "format": {
                    "type": "number-percent",
                    "decimalPlaces": {
                      "type": "max",
                      "places": 2
                    }
                  }
                }
              },
              "data": {
                "queryName": "main_query"
              }
            },
            "specExtensions": {
              "widgetBackgroundColor": {
                "light": "#BDDEED"
              },
              "widgetBorderColor": {
                "light": "#5364A0"
              }
            }
          },
          "position": {
            "x": 3,
            "y": 0,
            "width": 3,
            "height": 2
          }
        },
        {
          "widget": {
            "name": "fc5a5f42",
            "queries": [
              {
                "name": "main_query",
                "query": {
                  "datasetName": "e059b066",
                  "fields": [
                    {
                      "name": "yoy_pct",
                      "expression": "`yoy_pct`"
                    },
                    {
                      "name": "current_value",
                      "expression": "`current_value`"
                    }
                  ],
                  "disaggregated": true,
                  "orders": [
                    {
                      "direction": "DESC",
                      "expression": "`yoy_pct`"
                    }
                  ]
                }
              }
            ],
            "spec": {
              "frame": {
                "showTitle": true,
                "showDescription": true,
                "title": {
                  "value": "<span style=\"color: #07079D;\">Total Orders</span>",
                  "fields": []
                },
                "description": {
                  "value": "2013 | YoY vs 2012",
                  "fields": []
                }
              },
              "version": 2,
              "widgetType": "counter",
              "encodings": {
                "value": {
                  "fieldName": "current_value"
                },
                "period": {
                  "fieldName": "yoy_pct",
                  "style": {
                    "fontColor": {
                      "themeColorType": "visualizationColors",
                      "position": 3
                    }
                  },
                  "format": {
                    "type": "number-percent",
                    "decimalPlaces": {
                      "type": "max",
                      "places": 2
                    }
                  }
                }
              },
              "data": {
                "queryName": "main_query"
              }
            },
            "specExtensions": {
              "widgetBackgroundColor": {
                "light": "#FCE7F3"
              },
              "widgetBorderColor": {
                "light": "#D843A7"
              }
            }
          },
          "position": {
            "x": 6,
            "y": 0,
            "width": 3,
            "height": 2
          }
        },
        {
          "widget": {
            "name": "1bd0f838",
            "queries": [
              {
                "name": "main_query",
                "query": {
                  "datasetName": "051c5d4a",
                  "fields": [
                    {
                      "name": "yoy_pct",
                      "expression": "`yoy_pct`"
                    },
                    {
                      "name": "current_value",
                      "expression": "`current_value`"
                    }
                  ],
                  "disaggregated": true,
                  "orders": [
                    {
                      "direction": "DESC",
                      "expression": "`yoy_pct`"
                    }
                  ]
                }
              }
            ],
            "spec": {
              "frame": {
                "showTitle": true,
                "showDescription": true,
                "title": {
                  "value": "<span style=\"color: #07079D;\">Average Order Value (AOV)</span>",
                  "fields": []
                },
                "description": {
                  "value": "2013 | YoY vs 2012",
                  "fields": []
                }
              },
              "version": 2,
              "widgetType": "counter",
              "encodings": {
                "value": {
                  "fieldName": "current_value"
                },
                "period": {
                  "fieldName": "yoy_pct",
                  "style": {
                    "fontColor": "#CF1F2D"
                  },
                  "format": {
                    "type": "number-percent",
                    "decimalPlaces": {
                      "type": "max",
                      "places": 2
                    }
                  }
                }
              },
              "data": {
                "queryName": "main_query"
              }
            },
            "specExtensions": {
              "widgetBackgroundColor": {
                "light": "#BDDEED"
              },
              "widgetBorderColor": {
                "light": "#5364A0"
              }
            }
          },
          "position": {
            "x": 9,
            "y": 0,
            "width": 3,
            "height": 2
          }
        }
      ],
      "pageType": "PAGE_TYPE_CANVAS",
      "layoutVersion": "GRID_V1"
    }
  ],
  "uiSettings": {
    "theme": {
      "widgetHeaderAlignment": "ALIGNMENT_UNSPECIFIED"
    },
    "applyModeEnabled": false
  }
}
